[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/061.%20The%20Order%20Batching%20Problem/P61-Tier-5_executed.ipynb)

---


# 61. The Order Batching Problem
## Tier 5: An Integrated Digital Twin Simulation

This notebook builds a small, synthetic warehouse Digital Twin and uses it to illustrate how order batching can be evaluated as operating conditions change. The example combines a four-layer architecture, simulated sensor and workforce data, a weighted objective, a greedy initial assignment, and a short neighbor-search improvement loop.

### Purpose, inputs, and outputs

- **Purpose:** demonstrate how simulated operational data can inform batch assignments and scenario analysis.
- **Inputs:** the warehouse configuration, synthetic orders, picker availability, and the model parameters defined in the code.
- **Outputs:** simulated performance metrics, scenario summaries, and visualizations. The notebook does not connect to a real warehouse or external data source.
- **Solution method:** create an initial assignment by priority and picker load, evaluate it with a weighted objective, then accept better solutions generated by moving a batch between pickers. The loop runs for a fixed number of iterations; it does not guarantee a global optimum.

### Model assumptions and scope

The warehouse dimensions and counts of sensors, RFID tags, and picker wearables are illustrative configuration values. Sensor readings, order volumes, equipment conditions, and picker attributes are generated synthetically. The forecast is a random simulation, not a trained predictive model; no LSTM, CNN, or measured forecast accuracy is implemented. The optimization routine is a simplified heuristic, not a production-grade real-time optimizer. It models picking and travel time with fixed per-order estimates and does not represent warehouse aisle geometry, inventory locations, equipment capacity, or detailed labor rules. The stated data-volume rate is used only to illustrate a calculation.

### Mathematical model

Let $P$ be the set of available pickers, $O$ the set of orders, and $B_p$ the ordered list of batches assigned to picker $p$. Each order is assigned to one batch in this simplified model. The solution is scored by a weighted sum:

$$
J = \alpha T + \beta C + \gamma D.
$$

Here, $T$ is total estimated picking time in minutes, $C$ is estimated labor cost in dollars, and $D$ is the total delay penalty in dollars. The notebook uses $\alpha=0.4$, $\beta=0.3$, and $\gamma=0.3$. Since the components have different units and are not normalized, $J$ is an illustrative weighted score, not a quantity with a single physical unit. The weights express the chosen trade-off among the three components.

For picker $p$ and batch $b$, the estimated time is

$$
t_{pb} = \frac{2|b|}{e_p} + 1.5|b|,
$$

where $|b|$ is the number of orders in the batch and $e_p$ is picker efficiency. Picker efficiency is calculated from skill and fatigue as $e_p=s_p(1-0.5f_p)$. The first term estimates picking time; the second estimates travel time. In the code, these calculations are implemented by `_estimate_batch_picking_time`.

If order $i$ has relative deadline $d_i$ and estimated completion time $c_i$, its delay contribution is $10\max(0,c_i-d_i)$ dollars. Thus,

$$
D = \sum_{i\in O} 10\max(0,c_i-d_i).
$$

Completion times are estimated by adding each picker's batch times in assignment order. Orders within a batch are treated as completing together, at the end of that batch. The code implements this calculation in `_update_completion_times` and evaluates the penalty in `_calculate_delay_penalties`.

### How the search works

The initial solution sorts orders by priority and assigns them greedily to the currently least-loaded picker. A batch is extended until the existing load-based rule starts a new batch. The search then proposes a neighbor by moving one batch from one picker to another. A neighbor replaces the current best solution only when its objective is strictly lower. This is a simple local search: it can stop at a solution for which no tested move is accepted, even if another, untested assignment would be better.

For $n$ orders, $p$ available pickers, and $I$ neighbor iterations, the initial sorting and greedy assignment take approximately $O(n\log n + np)$ time. Evaluating each neighbor takes approximately $O(n+p)$ in this implementation, so the search takes approximately $O(I(n+p))$ additional time. These bounds describe the batching heuristic, not the full sensor simulation. The simulation also processes sensor records at each time step.

### What to examine

Follow how simulated data flows from the physical layer to the application layer, how the objective components are calculated, and how scenario conditions affect the resulting metrics. Treat the outputs as results of this synthetic model only; they do not establish real-world performance improvements or predictive accuracy.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds make the synthetic random streams reproducible across runs.
np.random.seed(42)
random.seed(42)

print("=== Order Batching: Integrated Digital Twin Simulation ===")
print("Required packages imported successfully.")

## 1. Represent the warehouse and its data layers

A Digital Twin is a digital representation connected to information about a physical system. Here, the four layers organize the simulation: physical devices and workers, data transmission, data storage, and application-level analytics. The classes below provide the data structures and simulated behavior for those layers. Because all measurements are generated in code, the result is a reproducible teaching simulation rather than a live digital twin.

In [ ]:
@dataclass
class IOSSensor:
    """Store the state and recent readings of a simulated IoT sensor."""

    id: str
    type: str
    location: Tuple[float, float]
    status: str = "active"
    last_update: datetime = field(default_factory=datetime.now)
    data_history: deque = field(default_factory=lambda: deque(maxlen=1000))


@dataclass
class PickerProfile:
    """Represent picker attributes used by the time and cost estimates."""

    id: int
    name: str
    skill_level: float
    current_fatigue: float = 0.0
    orders_completed: int = 0
    total_distance: float = 0.0
    efficiency_history: List[float] = field(default_factory=list)
    break_schedule: List[datetime] = field(default_factory=list)


@dataclass
class EquipmentStatus:
    """Store simulated equipment condition and predicted failure time."""

    id: str
    type: str
    location: Tuple[float, float]
    operational_status: str = "running"
    utilization: float = 0.0
    maintenance_score: float = 1.0
    predicted_failure_time: datetime = None


@dataclass
class OrderFlowData:
    """Hold one synthetic order-volume forecast and its model inputs."""

    timestamp: datetime
    order_volume: int
    order_types: Dict[str, int]
    peak_hours: bool = False
    seasonal_factor: float = 1.0
    external_factors: Dict[str, float] = field(default_factory=dict)


class DigitalTwinArchitecture:
    """Organize the simulated warehouse into four connected layers."""

    def __init__(self, warehouse_size=(100, 150), num_zones=10):
        self.warehouse_size = warehouse_size
        self.num_zones = num_zones

        # Physical layer: devices, equipment, and picker wearables.
        self.sensors = []
        self.rfid_tags = []
        self.picker_wearables = []
        self.equipment = []

        # Connectivity layer: simulated streams and transmission status.
        self.data_streams = defaultdict(list)
        self.network_status = "optimal"
        self.data_latency = 0.045

        # Data layer: containers for time series, topology, and live state.
        self.time_series_db = defaultdict(deque)
        self.graph_db = {}
        self.real_time_data = {}

        # Application layer: analytics, optimization, and dashboard state.
        self.ml_models = {}
        self.optimization_engine = None
        self.dashboard_data = {}

        self.current_time = datetime.now()
        self.simulation_speed = 1.0
        self.kpi_history = defaultdict(list)
        self.alert_system = []

        self._initialize_physical_layer()

    def _initialize_physical_layer(self):
        """Create the illustrative devices and picker profiles."""
        sensor_types = ["picker", "conveyor", "shelf", "entrance", "exit", "charging"]
        for i in range(847):
            sensor_type = np.random.choice(sensor_types)
            location = (
                np.random.uniform(0, self.warehouse_size[0]),
                np.random.uniform(0, self.warehouse_size[1]),
            )
            self.sensors.append(
                IOSSensor(id=f"sensor_{i}", type=sensor_type, location=location)
            )

        for i in range(12500):
            self.rfid_tags.append(
                {"id": f"rfid_{i}", "status": "active", "last_scan": self.current_time}
            )

        for i in range(156):
            self.picker_wearables.append(
                {
                    "id": f"wearable_{i}",
                    "picker_id": i,
                    "status": "active",
                    "battery_level": np.random.uniform(0.3, 1.0),
                    "location": (0, 0),
                }
            )
            self.real_time_data[f"picker_{i}"] = PickerProfile(
                id=i,
                name=f"Picker_{i}",
                skill_level=np.random.uniform(0.6, 1.0),
                current_fatigue=np.random.uniform(0.0, 0.3),
            )

        equipment_types = ["conveyor", "cart", "scanner", "charger"]
        for i in range(50):
            equipment_type = np.random.choice(equipment_types)
            location = (
                np.random.uniform(0, self.warehouse_size[0]),
                np.random.uniform(0, self.warehouse_size[1]),
            )
            self.equipment.append(
                EquipmentStatus(
                    id=f"eq_{i}",
                    type=equipment_type,
                    location=location,
                    maintenance_score=np.random.uniform(0.7, 1.0),
                )
            )

    def collect_sensor_data(self):
        """Generate and store one synthetic reading per active sensor."""
        timestamp = self.current_time
        for sensor in self.sensors:
            if sensor.status != "active":
                continue

            if sensor.type == "picker":
                value = np.random.normal(0.8, 0.1)
            elif sensor.type == "conveyor":
                value = np.random.normal(15.0, 2.0)
            elif sensor.type == "shelf":
                value = np.random.uniform(0.2, 0.9)
            elif sensor.type == "entrance":
                value = np.random.poisson(5)
            else:
                value = np.random.normal(0.5, 0.1)

            reading = {
                "timestamp": timestamp,
                "value": value,
                "sensor_id": sensor.id,
                "type": sensor.type,
                "location": sensor.location,
            }
            sensor.data_history.append(reading)
            self.data_streams[sensor.type].append(reading)
            sensor.last_update = timestamp

    def update_picker_performance(self):
        """Update synthetic fatigue, efficiency, and wearable battery state."""
        for wearable in self.picker_wearables:
            picker_id = wearable["picker_id"]
            profile = self.real_time_data[f"picker_{picker_id}"]

            hours_worked = (self.current_time.hour - 6) % 8
            fatigue_increase = hours_worked * 0.05 * 0.01
            profile.current_fatigue = min(
                1.0, profile.current_fatigue + fatigue_increase
            )
            efficiency = profile.skill_level * (1 - profile.current_fatigue * 0.5)
            profile.efficiency_history.append(efficiency)

            wearable["battery_level"] -= np.random.uniform(0.001, 0.005)
            if wearable["battery_level"] < 0.2:
                self.alert_system.append(
                    {
                        "timestamp": self.current_time,
                        "type": "low_battery",
                        "picker_id": picker_id,
                        "battery_level": wearable["battery_level"],
                    }
                )

    def predict_equipment_failure(self):
        """Apply a synthetic condition change and raise maintenance alerts."""
        for equipment in self.equipment:
            degradation = np.random.normal(0, 0.001)
            equipment.maintenance_score = max(
                0.0, equipment.maintenance_score - degradation
            )
            if equipment.maintenance_score < 0.3:
                hours_to_failure = np.random.exponential(24)
                equipment.predicted_failure_time = self.current_time + timedelta(
                    hours=hours_to_failure
                )
                self.alert_system.append(
                    {
                        "timestamp": self.current_time,
                        "type": "maintenance_alert",
                        "equipment_id": equipment.id,
                        "maintenance_score": equipment.maintenance_score,
                        "predicted_failure": equipment.predicted_failure_time,
                    }
                )

    def generate_order_flow_forecast(self, hours_ahead=4):
        """Generate a synthetic forecast from random and time-based factors."""
        base_volume = np.random.poisson(50)
        hour_factor = 1.0
        if 10 <= self.current_time.hour <= 12:
            hour_factor = 1.8
        elif 14 <= self.current_time.hour <= 16:
            hour_factor = 1.6
        elif 20 <= self.current_time.hour <= 22:
            hour_factor = 1.4

        seasonal_factor = 1.3 if self.current_time.weekday() >= 5 else 1.0
        weather_factor = np.random.uniform(0.8, 1.2)
        promotion_factor = 1.5 if np.random.random() < 0.1 else 1.0
        predicted_volume = int(
            base_volume
            * hour_factor
            * seasonal_factor
            * weather_factor
            * promotion_factor
        )

        return OrderFlowData(
            timestamp=self.current_time,
            order_volume=predicted_volume,
            order_types={
                "urgent": int(predicted_volume * 0.15),
                "normal": int(predicted_volume * 0.70),
                "bulk": int(predicted_volume * 0.15),
            },
            peak_hours=hour_factor > 1.2,
            seasonal_factor=seasonal_factor,
            external_factors={
                "weather": weather_factor,
                "promotion": promotion_factor,
            },
        )


print("Digital Twin data structures and simulation methods are ready.")

## 2. Define the objective and batching search

The objective turns a candidate assignment into three interpretable components: estimated time, estimated labor cost, and delay penalties. The search retains the same sequence throughout: build a greedy initial assignment, calculate its score, generate a neighbor by moving a batch, and keep the neighbor only if its score is lower. The time and cost estimates are deliberately simple so that the role of the objective and search can be inspected directly.

In [ ]:
class DynamicOptimizationEngine:
    """Evaluate and improve order batches with a weighted local search."""

    def __init__(self, digital_twin):
        self.digital_twin = digital_twin
        self.optimization_interval = 45
        self.last_optimization = None

        self.alpha = 0.4
        self.beta = 0.3
        self.gamma = 0.3

        self.current_batches = {}
        self.batch_assignments = {}
        self.optimization_history = []
        self.kpi_trends = defaultdict(list)

    def calculate_objective_function(self, batch_solution):
        """Return the weighted score and its three component values."""
        picker_assignments = batch_solution.get("picker_assignments", {})
        total_picking_time = sum(
            self._estimate_batch_picking_time(batch, picker_id)
            for picker_id, batch_list in picker_assignments.items()
            for batch in batch_list
        )
        labor_costs = self._calculate_labor_costs(picker_assignments)
        delay_penalties = self._calculate_delay_penalties(batch_solution)

        time_component = self.alpha * total_picking_time
        cost_component = self.beta * labor_costs
        delay_component = self.gamma * delay_penalties
        total_objective = time_component + cost_component + delay_component

        return {
            "total_objective": total_objective,
            "picking_time": total_picking_time,
            "labor_costs": labor_costs,
            "delay_penalties": delay_penalties,
            "breakdown": {
                "time_weight": time_component,
                "cost_weight": cost_component,
                "delay_weight": delay_component,
            },
        }

    def _estimate_batch_picking_time(self, batch, picker_id):
        """Estimate batch time using picker skill, fatigue, and order count."""
        profile = self.digital_twin.real_time_data.get(f"picker_{picker_id}")
        if profile is None:
            efficiency = 0.8
        else:
            efficiency = profile.skill_level * (1 - profile.current_fatigue * 0.5)

        base_picking_time = len(batch) * 2.0
        travel_time = len(batch) * 1.5
        return base_picking_time / efficiency + travel_time

    def _calculate_labor_costs(self, picker_assignments):
        """Estimate labor cost from work time, skill, and fatigue."""
        total_cost = 0.0
        for picker_id, batch_list in picker_assignments.items():
            profile = self.digital_twin.real_time_data.get(f"picker_{picker_id}")
            if profile is None:
                continue

            base_hourly_rate = 25.0
            skill_adjustment = 1.0 + (profile.skill_level - 0.7) * 0.5
            total_work_hours = sum(
                self._estimate_batch_picking_time(batch, picker_id)
                for batch in batch_list
            ) / 60.0
            fatigue_penalty = 1.0 + profile.current_fatigue * 0.2
            total_cost += (
                base_hourly_rate
                * skill_adjustment
                * total_work_hours
                * fatigue_penalty
            )
        return total_cost

    def _calculate_delay_penalties(self, batch_solution):
        """Penalize each order for estimated completion beyond its deadline."""
        total_penalty = 0.0
        deadlines = batch_solution.get("order_deadlines", {})
        completion_times = batch_solution.get("completion_times", {})

        for order_id, deadline in deadlines.items():
            if order_id not in completion_times:
                continue
            delay = completion_times[order_id] - deadline
            if delay > 0:
                total_penalty += delay * 10.0
        return total_penalty

    def _update_completion_times(self, solution):
        """Recompute completion estimates after assignments change."""
        completion_times = {}
        for picker_id, batch_list in solution["picker_assignments"].items():
            elapsed_time = 0.0
            for batch in batch_list:
                elapsed_time += self._estimate_batch_picking_time(batch, picker_id)
                for order in batch:
                    completion_times[order["id"]] = elapsed_time
        solution["completion_times"] = completion_times

    def optimize_batching(self, current_orders, picker_availability):
        """Run the greedy initialization and fixed-iteration neighbor search."""
        if current_orders and not picker_availability:
            raise ValueError("At least one picker is required when orders are present.")

        start_time = time.time()
        solution = self._generate_initial_solution(current_orders, picker_availability)
        objectives = self.calculate_objective_function(solution)
        best_solution = solution
        best_objectives = objectives

        for _ in range(10):
            neighbor_solution = self._generate_neighbor_solution(best_solution)
            neighbor_objectives = self.calculate_objective_function(neighbor_solution)
            if neighbor_objectives["total_objective"] < best_objectives["total_objective"]:
                best_solution = neighbor_solution
                best_objectives = neighbor_objectives

        optimization_time = time.time() - start_time
        self.current_batches = best_solution
        self.last_optimization = self.digital_twin.current_time
        self.optimization_history.append(
            {
                "timestamp": self.digital_twin.current_time,
                "optimization_time": optimization_time,
                "objective_value": best_objectives["total_objective"],
                "picking_time": best_objectives["picking_time"],
                "labor_costs": best_objectives["labor_costs"],
                "delay_penalties": best_objectives["delay_penalties"],
            }
        )
        return best_solution, best_objectives

    def _generate_initial_solution(self, orders, pickers):
        """Assign priority-sorted orders greedily to least-loaded pickers."""
        picker_assignments = {picker_id: [] for picker_id in pickers}
        picker_loads = {picker_id: 0 for picker_id in pickers}
        priority_rank = {"urgent": 3, "normal": 2, "bulk": 1}
        sorted_orders = sorted(
            orders,
            key=lambda order: priority_rank.get(order.get("priority", "normal"), 0),
            reverse=True,
        )

        for order in sorted_orders:
            least_loaded_picker = min(pickers, key=lambda picker: picker_loads[picker])
            if (
                not picker_assignments[least_loaded_picker]
                or picker_loads[least_loaded_picker] > 20
            ):
                picker_assignments[least_loaded_picker].append([order])
            else:
                picker_assignments[least_loaded_picker][-1].append(order)
            picker_loads[least_loaded_picker] += order.get("volume", 1)

        solution = {
            "picker_assignments": picker_assignments,
            "order_deadlines": {
                order["id"]: order.get("deadline", 100) for order in orders
            },
            "completion_times": {},
        }
        self._update_completion_times(solution)
        return solution

    def _generate_neighbor_solution(self, current_solution):
        """Create a neighbor by moving one batch between two pickers."""
        new_solution = {
            "picker_assignments": {
                picker_id: [batch.copy() for batch in batches]
                for picker_id, batches in current_solution["picker_assignments"].items()
            },
            "order_deadlines": current_solution["order_deadlines"].copy(),
            "completion_times": current_solution["completion_times"].copy(),
        }
        pickers = list(new_solution["picker_assignments"])
        if len(pickers) > 1 and np.random.random() < 0.5:
            source_picker = np.random.choice(pickers)
            target_picker = np.random.choice(
                [picker for picker in pickers if picker != source_picker]
            )
            if new_solution["picker_assignments"][source_picker]:
                batch = new_solution["picker_assignments"][source_picker].pop()
                new_solution["picker_assignments"][target_picker].append(batch)

        self._update_completion_times(new_solution)
        return new_solution


print("Weighted objective and batching search are defined.")

## 3. Create the illustrative warehouse instance

The instance uses the stated example configuration: a warehouse represented as 800 by 1,200 feet, 40 zones, 847 sensors, 12,500 RFID tags, 156 picker wearables, and 50 equipment units. These values initialize synthetic objects; they are not evidence about an operating facility. The optimizer's 45-second interval is a configuration setting. The later simulation advances in five-minute steps, so it does not resolve or test a 45-second scheduling interval.

In [ ]:
def create_amazon_seattle_digital_twin():
    """Create the illustrative warehouse instance and its optimizer."""
    print("=== Creating the Illustrative Warehouse Digital Twin ===")
    digital_twin = DigitalTwinArchitecture(
        warehouse_size=(800, 1200),
        num_zones=40,
    )
    optimization_engine = DynamicOptimizationEngine(digital_twin)
    digital_twin.optimization_engine = optimization_engine

    print(f"Warehouse dimensions: {digital_twin.warehouse_size[0]} by {digital_twin.warehouse_size[1]} feet")
    print(f"Zones: {digital_twin.num_zones}")
    print(f"Sensors: {len(digital_twin.sensors)}")
    print(f"RFID tags: {len(digital_twin.rfid_tags)}")
    print(f"Picker wearables: {len(digital_twin.picker_wearables)}")
    print(f"Equipment units: {len(digital_twin.equipment)}")
    return digital_twin, optimization_engine


digital_twin, opt_engine = create_amazon_seattle_digital_twin()

print("\nFour-layer architecture:")
print(f"Physical: {len(digital_twin.sensors)} sensors and {len(digital_twin.equipment)} equipment units")
print(f"Connectivity: {digital_twin.data_latency:.3f} seconds simulated latency; {digital_twin.network_status} status")
print("Data: containers for time-series records, topology, and real-time state")
print("Application: synthetic forecast and weighted batching search")
print(f"Configured optimization interval: {opt_engine.optimization_interval} seconds")
print(f"Objective weights: time={opt_engine.alpha}, labor cost={opt_engine.beta}, delay={opt_engine.gamma}")

## 4. Simulate operation over time

Each five-minute step advances the simulated clock, samples active sensors, updates picker and equipment states, generates a synthetic volume forecast, and creates a separate synthetic order count. Separating forecast and actual simulated volume avoids defining forecast accuracy by comparing a prediction with itself. The absolute percentage-style accuracy computed below is still only a synthetic proxy: the forecast and actual values are generated by this notebook, not measured from a forecasting system. The batching search runs once per five-minute step.

In [ ]:
def simulate_digital_twin_operation(digital_twin, opt_engine, duration_hours=8):
    """Run the synthetic warehouse simulation at five-minute intervals."""
    print(f"=== Simulating {duration_hours} hours of operation ===")
    time_step_minutes = 5
    total_steps = (duration_hours * 60) // time_step_minutes
    performance_metrics = {
        "timestamps": [],
        "order_volume": [],
        "picking_time": [],
        "labor_costs": [],
        "delay_penalties": [],
        "prediction_accuracy": [],
        "equipment_alerts": [],
        "picker_alerts": [],
        "data_volume_mb": [],
    }

    for step in range(total_steps):
        digital_twin.current_time += timedelta(minutes=time_step_minutes)
        current_time = digital_twin.current_time

        digital_twin.collect_sensor_data()
        digital_twin.update_picker_performance()
        digital_twin.predict_equipment_failure()
        forecast = digital_twin.generate_order_flow_forecast()

        # The actual count is sampled separately from the forecast value.
        actual_order_volume = int(np.random.poisson(forecast.order_volume))
        current_orders = []
        for order_index in range(actual_order_volume):
            current_orders.append(
                {
                    "id": f"order_{step}_{order_index}",
                    "volume": int(np.random.randint(1, 5)),
                    "priority": str(
                        np.random.choice(
                            ["urgent", "normal", "bulk"], p=[0.15, 0.70, 0.15]
                        )
                    ),
                    "deadline": int(np.random.randint(30, 120)),
                    "arrival_time": current_time,
                }
            )

        available_pickers = [
            wearable["picker_id"]
            for wearable in digital_twin.picker_wearables
            if wearable["battery_level"] > 0.3 and wearable["status"] == "active"
        ]
        solution, objectives = opt_engine.optimize_batching(
            current_orders, available_pickers
        )

        denominator = max(actual_order_volume, forecast.order_volume, 1)
        accuracy_proxy = 1.0 - abs(
            actual_order_volume - forecast.order_volume
        ) / denominator
        recent_alerts = [
            alert
            for alert in digital_twin.alert_system
            if current_time - timedelta(minutes=time_step_minutes)
            < alert["timestamp"]
            <= current_time
        ]

        performance_metrics["timestamps"].append(current_time)
        performance_metrics["order_volume"].append(actual_order_volume)
        performance_metrics["picking_time"].append(objectives["picking_time"])
        performance_metrics["labor_costs"].append(objectives["labor_costs"])
        performance_metrics["delay_penalties"].append(objectives["delay_penalties"])
        performance_metrics["prediction_accuracy"].append(accuracy_proxy)
        performance_metrics["equipment_alerts"].append(
            sum(alert["type"] == "maintenance_alert" for alert in recent_alerts)
        )
        performance_metrics["picker_alerts"].append(
            sum(alert["type"] == "low_battery" for alert in recent_alerts)
        )

        # Allocate the illustrative 2.3 TiB daily data rate evenly across 288 five-minute steps.
        data_volume_mb = (2.3 * 1024**2) / (24 * 60 / time_step_minutes)
        performance_metrics["data_volume_mb"].append(data_volume_mb)

        if step % 20 == 0:
            progress = 100 * (step + 1) / total_steps
            print(
                f"Progress: {progress:.1f}% | "
                f"time: {current_time.strftime('%H:%M')} | "
                f"orders: {actual_order_volume} | "
                f"available pickers: {len(available_pickers)}"
            )

    print("Simulation completed.")
    return performance_metrics


performance_metrics = simulate_digital_twin_operation(
    digital_twin, opt_engine, duration_hours=8
)

## 5. Summarize the simulated performance

The analysis reports averages over the recorded five-minute periods, along with the total illustrative data volume and alert counts. The picking-time comparison uses an explicitly defined synthetic baseline: baseline time is set to 1.28 times the simulated time. This construction yields a 21.875% reduction relative to that baseline; it is not an independently measured improvement and should not be interpreted as evidence about a real warehouse.

In [ ]:
def analyze_simulation_results(performance_metrics, opt_engine):
    """Calculate summary statistics from the supplied simulation records."""
    if not performance_metrics["timestamps"]:
        raise ValueError("The simulation must contain at least one time step.")

    avg_order_volume = float(np.mean(performance_metrics["order_volume"]))
    avg_picking_time = float(np.mean(performance_metrics["picking_time"]))
    avg_labor_costs = float(np.mean(performance_metrics["labor_costs"]))
    avg_delay_penalties = float(np.mean(performance_metrics["delay_penalties"]))
    avg_prediction_accuracy = float(
        np.mean(performance_metrics["prediction_accuracy"])
    )
    total_data_tib = sum(performance_metrics["data_volume_mb"]) / 1024**2

    static_baseline_time = avg_picking_time * 1.28
    if static_baseline_time:
        time_reduction = (
            (static_baseline_time - avg_picking_time) / static_baseline_time * 100
        )
    else:
        time_reduction = 0.0

    total_equipment_alerts = sum(performance_metrics["equipment_alerts"])
    total_picker_alerts = sum(performance_metrics["picker_alerts"])
    optimization_updates = len(performance_metrics["timestamps"])

    print("=== Synthetic Simulation Summary ===")
    print(f"Average orders per five-minute period: {avg_order_volume:.1f}")
    print(f"Average estimated picking time: {avg_picking_time:.2f} minutes")
    print(f"Average estimated labor cost: ${avg_labor_costs:.2f}")
    print(f"Average delay penalty: ${avg_delay_penalties:.2f}")
    print(f"Mean synthetic forecast proxy: {avg_prediction_accuracy:.1%}")
    print(f"Illustrative data processed: {total_data_tib:.3f} TiB")
    print(f"Reduction versus the constructed baseline: {time_reduction:.1f}%")
    print(f"Equipment alerts: {total_equipment_alerts}")
    print(f"Picker battery alerts: {total_picker_alerts}")
    print(f"Optimization calls: {optimization_updates}")
    print(
        f"Simulation step: 5 minutes; configured optimizer interval: "
        f"{opt_engine.optimization_interval} seconds"
    )

    return {
        "avg_order_volume": avg_order_volume,
        "avg_picking_time": avg_picking_time,
        "avg_labor_costs": avg_labor_costs,
        "avg_delay_penalties": avg_delay_penalties,
        "avg_prediction_accuracy": avg_prediction_accuracy,
        "time_reduction": time_reduction,
        "total_alerts": total_equipment_alerts + total_picker_alerts,
        "total_data_tib": total_data_tib,
        "optimization_updates": optimization_updates,
    }


analysis_results = analyze_simulation_results(performance_metrics, opt_engine)

## 6. Compare what-if scenarios

Each scenario changes the order-volume multiplier or the fraction of pickers available, then runs the same batching search. The order-surge multiplier of 2.3 means the simulated demand is 2.3 times its sampled base volume, or 130% above that base. For computational control, at most 200 orders are created per simulated minute in this experiment. Completion rate is the fraction of the scenario's generated demand represented by assigned orders; it is not an on-time service rate. The reported adaptation time is a simple proxy: the first minute at which that completion fraction exceeds 90%.

In [ ]:
def simulate_what_if_scenarios(digital_twin, opt_engine):
    """Evaluate the same batching search under four synthetic conditions."""
    scenarios = {
        "order_surge": {
            "volume_multiplier": 2.3,
            "description": "Demand at 2.3 times the sampled base volume",
        },
        "equipment_failure": {
            "equipment_availability": 0.7,
            "description": "30% equipment unavailability (not directly modeled)",
        },
        "picker_shortage": {
            "picker_availability": 0.6,
            "description": "40% picker unavailability",
        },
        "normal": {
            "volume_multiplier": 1.0,
            "equipment_availability": 1.0,
            "picker_availability": 1.0,
            "description": "Reference scenario",
        },
    }

    scenario_results = {}
    original_time = digital_twin.current_time

    try:
        for scenario_name, parameters in scenarios.items():
            print(f"\nScenario: {parameters['description']}")
            scenario_metrics = {
                "picking_time": [],
                "completion_rate": [],
                "delay_penalties": [],
                "adaptation_time": None,
            }

            for minute in range(120):
                digital_twin.current_time += timedelta(minutes=1)
                volume_multiplier = parameters.get("volume_multiplier", 1.0)
                picker_availability = parameters.get("picker_availability", 1.0)
                base_volume = int(np.random.poisson(50))
                order_volume = int(base_volume * volume_multiplier)
                current_orders = []

                for order_index in range(min(order_volume, 200)):
                    current_orders.append(
                        {
                            "id": f"order_{scenario_name}_{minute}_{order_index}",
                            "volume": int(np.random.randint(1, 5)),
                            "priority": str(
                                np.random.choice(
                                    ["urgent", "normal", "bulk"],
                                    p=[0.2, 0.6, 0.2],
                                )
                            ),
                            "deadline": int(np.random.randint(30, 120)),
                        }
                    )

                available_pickers = [
                    wearable["picker_id"]
                    for wearable in digital_twin.picker_wearables
                    if wearable["battery_level"] > 0.3
                    and wearable["status"] == "active"
                    and np.random.random() < picker_availability
                ]

                if current_orders and available_pickers:
                    solution, objectives = opt_engine.optimize_batching(
                        current_orders, available_pickers
                    )
                    assigned_orders = sum(
                        len(batch)
                        for batches in solution["picker_assignments"].values()
                        for batch in batches
                    )
                    completion_rate = assigned_orders / max(1, order_volume)
                    scenario_metrics["picking_time"].append(
                        objectives["picking_time"]
                    )
                    scenario_metrics["completion_rate"].append(completion_rate)
                    scenario_metrics["delay_penalties"].append(
                        objectives["delay_penalties"]
                    )
                    if (
                        scenario_metrics["adaptation_time"] is None
                        and completion_rate >= 0.9
                    ):
                        scenario_metrics["adaptation_time"] = minute

            scenario_results[scenario_name] = {
                "avg_picking_time": float(
                    np.mean(scenario_metrics["picking_time"])
                )
                if scenario_metrics["picking_time"]
                else 0.0,
                "avg_completion_rate": float(
                    np.mean(scenario_metrics["completion_rate"])
                )
                if scenario_metrics["completion_rate"]
                else 0.0,
                "avg_delay_penalties": float(
                    np.mean(scenario_metrics["delay_penalties"])
                )
                if scenario_metrics["delay_penalties"]
                else 0.0,
                "adaptation_time": scenario_metrics["adaptation_time"],
            }
            result = scenario_results[scenario_name]
            print(f"  Mean estimated picking time: {result['avg_picking_time']:.2f} minutes")
            print(f"  Mean assigned-demand fraction: {result['avg_completion_rate']:.1%}")
            print(f"  Mean delay penalty: ${result['avg_delay_penalties']:.2f}")
            adaptation_time = result["adaptation_time"]
            print(
                f"  First minute at or above the 90% threshold: {adaptation_time}"
                if adaptation_time is not None
                else "  The 90% threshold was not reached"
            )
    finally:
        digital_twin.current_time = original_time

    return scenario_results


scenario_results = simulate_what_if_scenarios(digital_twin, opt_engine)

print("\nScenario differences relative to the reference scenario:")
normal_result = scenario_results["normal"]
for scenario_name, result in scenario_results.items():
    if scenario_name == "normal":
        continue
    if normal_result["avg_picking_time"]:
        time_impact = (
            (result["avg_picking_time"] - normal_result["avg_picking_time"])
            / normal_result["avg_picking_time"]
            * 100
        )
    else:
        time_impact = 0.0
    print(f"{scenario_name.replace('_', ' ').title()}: picking-time difference {time_impact:+.1f}%")

## 7. Visualize the simulation and scenario results

The panels show order volume alongside estimated picking time, the synthetic forecast proxy over time, scenario-level picking time and assigned-demand fraction, and alert counts. Dual axes are used where quantities have different units; the axis labels and legends identify which series belongs to each scale. The dashed forecast reference line is a visual reference only, not a measured target.

In [ ]:
def create_digital_twin_visualizations(performance_metrics, scenario_results):
    """Plot the recorded metrics and scenario summaries with labeled axes."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle("Synthetic Order Batching Digital Twin Analysis", fontsize=15)

    period_index = np.arange(len(performance_metrics["order_volume"]))

    ax1 = axes[0, 0]
    ax1.bar(
        period_index,
        performance_metrics["order_volume"],
        alpha=0.6,
        color="skyblue",
        label="Orders",
    )
    ax1.set_xlabel("Five-minute simulation period")
    ax1.set_ylabel("Order count")
    ax1_twin = ax1.twinx()
    ax1_twin.plot(
        period_index,
        performance_metrics["picking_time"],
        color="darkred",
        linewidth=1.8,
        label="Estimated picking time",
    )
    ax1_twin.set_ylabel("Estimated picking time (minutes)")
    ax1.set_title("Orders and Estimated Picking Time")
    handles_1, labels_1 = ax1.get_legend_handles_labels()
    handles_2, labels_2 = ax1_twin.get_legend_handles_labels()
    ax1.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left")
    ax1.grid(True, alpha=0.25)

    ax2 = axes[0, 1]
    ax2.plot(
        period_index,
        performance_metrics["prediction_accuracy"],
        color="green",
        linewidth=1.8,
        marker="o",
        markersize=3,
        label="Synthetic proxy",
    )
    ax2.axhline(
        y=0.913,
        color="gray",
        linestyle="--",
        alpha=0.7,
        label="Illustrative 91.3% reference",
    )
    ax2.set_xlabel("Five-minute simulation period")
    ax2.set_ylabel("Forecast proxy (fraction)")
    ax2.set_ylim(0, 1.05)
    ax2.set_title("Synthetic Order-Volume Forecast Proxy")
    ax2.legend()
    ax2.grid(True, alpha=0.25)

    ax3 = axes[1, 0]
    scenario_names = list(scenario_results)
    positions = np.arange(len(scenario_names))
    width = 0.35
    picking_times = [
        scenario_results[name]["avg_picking_time"] for name in scenario_names
    ]
    completion_rates = [
        scenario_results[name]["avg_completion_rate"] for name in scenario_names
    ]
    ax3.bar(
        positions - width / 2,
        picking_times,
        width,
        color="lightcoral",
        alpha=0.8,
        label="Picking time (minutes)",
    )
    ax3_twin = ax3.twinx()
    ax3_twin.bar(
        positions + width / 2,
        completion_rates,
        width,
        color="lightgreen",
        alpha=0.8,
        label="Assigned-demand fraction",
    )
    ax3.set_xlabel("Scenario")
    ax3.set_ylabel("Mean estimated picking time (minutes)")
    ax3_twin.set_ylabel("Mean assigned-demand fraction")
    ax3_twin.set_ylim(0, 1.05)
    ax3.set_xticks(positions)
    ax3.set_xticklabels(
        [name.replace("_", "\n").title() for name in scenario_names],
        rotation=0,
    )
    ax3.set_title("What-If Scenario Comparison")
    handles_3a, labels_3a = ax3.get_legend_handles_labels()
    handles_3b, labels_3b = ax3_twin.get_legend_handles_labels()
    ax3.legend(handles_3a + handles_3b, labels_3a + labels_3b, loc="upper right")

    ax4 = axes[1, 1]
    ax4.bar(
        period_index,
        performance_metrics["equipment_alerts"],
        label="Equipment alerts",
        color="orange",
        alpha=0.75,
    )
    ax4.bar(
        period_index,
        performance_metrics["picker_alerts"],
        bottom=performance_metrics["equipment_alerts"],
        label="Picker battery alerts",
        color="purple",
        alpha=0.75,
    )
    ax4.set_xlabel("Five-minute simulation period")
    ax4.set_ylabel("Alert count")
    ax4.set_title("Simulated Alerts")
    ax4.legend()
    ax4.grid(True, alpha=0.25)

    fig.tight_layout()
    plt.show()
    return fig


visualization_fig = create_digital_twin_visualizations(
    performance_metrics, scenario_results
)
print("Simulation visualizations are ready.")

## 8. Inspect architecture and computational scope

The architecture counts below describe the initialized objects and the records created during this run. For the batching heuristic, let $n$ be the number of orders in a call, $p$ the number of available pickers, and $I=10$ the fixed neighbor-search iterations. The initial assignment is approximately $O(n\log n+np)$, and evaluating the fixed number of neighbors is approximately $O(I(n+p))$. Sensor collection adds work proportional to the number of active sensors per simulation step. The bounded sensor history limits retained readings per sensor, while the stream lists and alert list grow as the simulation appends records.

In [ ]:
print("=== Digital Twin Simulation Inspection ===")
print("\nFour-layer architecture:")
print(f"Physical layer: {len(digital_twin.sensors)} sensors, {len(digital_twin.rfid_tags)} RFID tags")
print(f"Picker layer: {len(digital_twin.picker_wearables)} wearables and picker profiles")
print(f"Equipment layer: {len(digital_twin.equipment)} equipment units")
print(f"Connectivity latency setting: {digital_twin.data_latency * 1000:.1f} milliseconds")
print(f"Sensor stream types observed: {len(digital_twin.data_streams)}")
print(f"Recorded optimization periods: {len(performance_metrics['timestamps'])}")
print(f"Illustrative data volume: {analysis_results['total_data_tib']:.3f} TiB")
print("\nBatching-search scope:")
print("Initial assignment: priority sort followed by least-loaded-picker assignment")
print("Neighbor move: transfer one batch between two pickers")
print("Neighbor iterations per call: 10")
print("Acceptance rule: keep a neighbor only when its weighted objective is lower")
print("\nModel limitations:")
print("Sensor values, forecasts, fatigue, equipment degradation, and orders are synthetic.")
print("Equipment availability is not applied to batch feasibility in the scenario routine.")
print("Picker skill and fatigue affect estimated time; travel is a fixed per-order estimate.")
print("The local search does not guarantee a globally optimal batching plan.")

## 9. Interpret the results and limitations

The reported values are computed from the generated records when the notebook runs; they are not embedded benchmark results. The scenario comparison isolates how the specified synthetic demand and picker-availability changes affect this particular heuristic and its simplified objective. Equipment availability is included as scenario metadata but is not used to remove equipment or constrain assignments, so that scenario does not test equipment-failure effects on batching. Likewise, the forecast proxy is not a validation metric for a trained prediction model. Any operational conclusion would require validated input data, realistic travel and capacity models, and an independently justified comparison baseline.

In [ ]:
print("=== Summary ===")
print("This run combines four-layer warehouse data structures with a synthetic simulation.")
print(
    f"The simulation recorded {analysis_results['optimization_updates']} "
    "batching calls at five-minute time steps."
)
print(
    f"Mean estimated picking time: "
    f"{analysis_results['avg_picking_time']:.2f} minutes per simulated period."
)
print(
    f"Mean synthetic forecast proxy: "
    f"{analysis_results['avg_prediction_accuracy']:.1%}."
)
print(
    f"Scenario results include {len(scenario_results)} conditions; "
    "their assigned-demand fractions are not on-time service rates."
)
print(
    "Interpret all metrics as outputs of the stated assumptions and random "
    "simulation, not as measured warehouse performance."
)
print("\nThe central method is the weighted objective evaluated after greedy assignment and fixed-iteration neighbor search.")